# 16 — Spatial Value-Add Atlas for EWS

## What this notebook adds

This notebook is designed to **strengthen your ISEF story without overclaiming**.

It asks a sharper question than “does my model work?”:

> **Does EWS add predictive value beyond climate-only drivers, and is that added value geographically concentrated in California’s upwelling core?**

That matters because a judge can reasonably say:
- maybe SST/upwelling already explain the result,
- maybe AUC alone is not enough,
- maybe central California only looks strong because of geography.

This notebook answers those critiques by doing a **site-by-site held-out comparison**:

- **Climate-only model:** heat + upwelling
- **Full model:** EWS + heat + upwelling + heat×EWS
- **Evaluation:** same held-out site, same quarters, paired block-bootstrap ΔAUC
- **Main visual:** a **California map** showing where EWS truly adds value
- **Extra rigor:** pooled calibration / AP / Brier + optional low-*n* robustness

## Preferred input

Use a prebuilt site-feature file, ideally from your site-level notebook.  
The easiest format is a pickle created from your site feature dictionary:

```python
pd.to_pickle(site_data, "all_site_features.pkl")
```

where `site_data` is a dict like:

```python
{
    "Santa Cruz": DataFrame(...),
    "Bodega Bay": DataFrame(...),
    ...
}
```

Each site DataFrame should contain at least:
- `ews_composite`
- `heat_lag4`
- `onset`

Recommended columns:
- `upwelling`
- `heat_x_ews`
- `suppressed`
- `kelp_smooth`

## Why this is useful for ISEF

If the result holds, this notebook gives you a stronger claim:

> **“EWS improves prediction beyond climate-only drivers, and that improvement is spatially concentrated in central California.”**

That is more rigorous and more memorable than only reporting a single AUC.

In [1]:
# ============================================================
# CELL 1 — IMPORTS & CONFIG
# ============================================================
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm, Normalize
from scipy.stats import mannwhitneyu
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 1.2,
    "figure.dpi": 150,
})

# ---- Input path: update this if needed ----
# Preferred file: a pickle dict {site_name -> DataFrame}
CANDIDATE_INPUTS = [
    Path("./all_site_features.pkl"),
    Path("./site_data.pkl"),
    Path("./site_features.pkl"),
    Path("./1_DATA/processed/all_site_features.pkl"),
    Path("./1_DATA/processed/site_features.pkl"),
    Path("./all_site_features.csv"),
    Path("./site_features.csv"),
]
SITE_DATA_PATH = next((p for p in CANDIDATE_INPUTS if p.exists()), CANDIDATE_INPUTS[0])

# ---- Output folder ----
OUTPUT_DIR = Path("./16_spatial_value_add_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Evaluation settings ----
CV_MODE = "neighbor"       # "neighbor" or "loso"
NEIGHBOR_DEG = 3.0         # used only if CV_MODE == "neighbor"
MIN_TRAIN_ONSETS = 2
MIN_TEST_ROWS = 12
BLOCK_LEN = 4
B_BOOT = 1000
THRESHOLD = 0.35           # for strict next-quarter warning metrics only
LOW_N_MIN_EVENTS = 3       # robustness threshold
RUN_LOW_N_ROBUSTNESS = True

BASELINE_FEATURES_PREF = ["heat_lag4", "upwelling"]
FULL_FEATURES_PREF = ["ews_composite", "heat_lag4", "upwelling", "heat_x_ews"]
TARGET = "onset"

print("Config ready")
print(f"  SITE_DATA_PATH : {SITE_DATA_PATH}")
print(f"  OUTPUT_DIR     : {OUTPUT_DIR.resolve()}")
print(f"  CV_MODE        : {CV_MODE}")
if CV_MODE == "neighbor":
    print(f"  NEIGHBOR_DEG   : {NEIGHBOR_DEG}° latitude")
print(f"  B_BOOT         : {B_BOOT}  (increase to 2000 for final figures if runtime allows)")
print(f"  THRESHOLD      : {THRESHOLD}")

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# ============================================================
# CELL 2 — SITE METADATA (19 California sites)
# Update names only if your site labels differ
# ============================================================
SITE_META_RAW = [
    # name,                lat,   lon,    region_color, central_ca, is_region
    ("Crescent City",      41.8, -124.2, "#1f77b4", False, False),
    ("Cape Mendocino",     40.5, -124.4, "#1f77b4", False, False),
    ("NorCal Region",      39.0, -123.8, "#1f77b4", False, True),

    ("Bodega Bay",         38.4, -123.1, "#d6604d", True,  False),
    ("Point Reyes",        38.0, -122.9, "#d6604d", True,  False),
    ("Half Moon Bay",      37.5, -122.5, "#d6604d", True,  False),
    ("Santa Cruz",         37.0, -122.1, "#d6604d", True,  False),
    ("MidCal Region",      36.8, -122.0, "#d6604d", True,  True),

    ("Point Sur",          36.4, -121.9, "#9467bd", False, False),
    ("Cambria",            35.6, -121.1, "#9467bd", False, False),
    ("BigSur Region",      35.4, -121.0, "#9467bd", False, True),
    ("Morro Bay",          35.4, -120.9, "#9467bd", False, False),

    ("Point Conception",   34.5, -120.5, "#2ca02c", False, False),
    ("Santa Barbara",      34.4, -119.7, "#2ca02c", False, False),
    ("SoCal Region",       34.3, -119.5, "#2ca02c", False, True),
    ("Ventura",            34.2, -119.3, "#2ca02c", False, False),
    ("Palos Verdes",       33.8, -118.4, "#2ca02c", False, False),
    ("Laguna Beach",       33.5, -117.8, "#2ca02c", False, False),
    ("San Diego",          32.8, -117.3, "#2ca02c", False, False),
]

SITE_META = pd.DataFrame(
    SITE_META_RAW,
    columns=["site", "lat", "lon", "color", "central", "is_region"]
)

ALIASES = {
    "norcal": "NorCal Region",
    "midcal": "MidCal Region",
    "socal": "SoCal Region",
    "bigsur": "BigSur Region",
    "big sur": "BigSur Region",
    "point reyes national seashore": "Point Reyes",
}

def normalize_site_name(name):
    s = str(name).strip()
    s_clean = " ".join(s.replace("_", " ").split())
    lower = s_clean.lower()
    if lower in ALIASES:
        return ALIASES[lower]
    known = SITE_META["site"].tolist()
    if s_clean in known:
        return s_clean
    for k in known:
        if s_clean.lower() == k.lower():
            return k
    return s_clean

SITE_META["site_norm"] = SITE_META["site"].map(normalize_site_name)
SITE_META = SITE_META.drop_duplicates("site_norm").set_index("site_norm", drop=False)

print("Known sites:")
print(SITE_META[["site", "lat", "lon", "central", "is_region"]].to_string(index=False))

In [ ]:
# ============================================================
# CELL 3 — LOAD + VALIDATE SITE FEATURE DATA
# Supports:
#   1) pickle dict {site -> DataFrame}
#   2) pickle DataFrame with a site column
#   3) csv/parquet long table with a site column
# ============================================================
def _prep_time_index(df):
    df = df.copy()
    if isinstance(df.index, pd.DatetimeIndex):
        idx = pd.to_datetime(df.index)
    else:
        time_candidates = ["time", "date", "datetime", "quarter", "timestamp"]
        found = None
        for c in time_candidates:
            if c in df.columns:
                found = c
                break
        if found is None:
            raise ValueError(
                "No DatetimeIndex and no time/date column found. "
                "Add a time column or set the index before saving."
            )
        idx = pd.to_datetime(df[found], errors="coerce")
        df = df.drop(columns=[found])
    if idx.isna().all():
        raise ValueError("Time parsing failed for all rows.")
    df.index = pd.to_datetime(idx).tz_localize(None)
    df = df[~df.index.isna()].sort_index()
    return df

def _enrich_features(df):
    df = df.copy()

    if "ews_composite" not in df.columns and {"ar1_z", "var_z"}.issubset(df.columns):
        df["ews_composite"] = (df["ar1_z"] + df["var_z"]) / 2.0

    if "heat_x_ews" not in df.columns and {"heat_lag4", "ews_composite"}.issubset(df.columns):
        df["heat_x_ews"] = df["heat_lag4"] * df["ews_composite"]

    if TARGET not in df.columns and "suppressed" in df.columns:
        s = pd.Series(df["suppressed"]).fillna(0).astype(int)
        df[TARGET] = ((s == 1) & (s.shift(1).fillna(0) == 0)).astype(int)

    return df

def _split_long_df(df):
    site_col = None
    for c in ["site", "site_name", "name", "site_id"]:
        if c in df.columns:
            site_col = c
            break
    if site_col is None:
        raise ValueError(
            "Long-form DataFrame must contain one of: "
            "site, site_name, name, site_id"
        )
    out = {}
    for raw_name, sdf in df.groupby(site_col):
        nm = normalize_site_name(raw_name)
        sdf = sdf.drop(columns=[site_col]).copy()
        sdf = _prep_time_index(sdf)
        sdf = _enrich_features(sdf)
        out[nm] = sdf
    return out

def load_site_data(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Input file not found: {path}\n\n"
            "Recommended fix:\n"
            "  pd.to_pickle(site_data, 'all_site_features.pkl')\n"
            "from your site-level notebook, then point SITE_DATA_PATH to it."
        )

    suffix = path.suffix.lower()
    if suffix in [".pkl", ".pickle"]:
        obj = pd.read_pickle(path)
        if isinstance(obj, dict):
            out = {}
            for raw_name, df in obj.items():
                nm = normalize_site_name(raw_name)
                sdf = _prep_time_index(pd.DataFrame(df))
                sdf = _enrich_features(sdf)
                out[nm] = sdf
            return out
        if isinstance(obj, pd.DataFrame):
            return _split_long_df(obj)
        raise TypeError(f"Unsupported pickle content: {type(obj)}")

    if suffix == ".csv":
        df = pd.read_csv(path)
        return _split_long_df(df)

    if suffix == ".parquet":
        df = pd.read_parquet(path)
        return _split_long_df(df)

    raise ValueError(f"Unsupported input type: {suffix}")

site_data_raw = load_site_data(SITE_DATA_PATH)

site_data = {}
site_rows = []

for site_name, df in site_data_raw.items():
    df = df.copy().sort_index()

    lat_val = np.nan
    lon_val = np.nan
    if site_name in SITE_META.index:
        lat_val = float(SITE_META.loc[site_name, "lat"])
        lon_val = float(SITE_META.loc[site_name, "lon"])
        color = SITE_META.loc[site_name, "color"]
        central = bool(SITE_META.loc[site_name, "central"])
        is_region = bool(SITE_META.loc[site_name, "is_region"])
    else:
        lat_candidates = [c for c in ["lat", "latitude"] if c in df.columns]
        lon_candidates = [c for c in ["lon", "longitude"] if c in df.columns]
        if lat_candidates:
            temp = pd.to_numeric(df[lat_candidates[0]], errors="coerce").dropna()
            lat_val = temp.iloc[0] if len(temp) else np.nan
        if lon_candidates:
            temp = pd.to_numeric(df[lon_candidates[0]], errors="coerce").dropna()
            lon_val = temp.iloc[0] if len(temp) else np.nan
        central = bool(36.5 <= lat_val <= 38.5) if pd.notna(lat_val) else False
        is_region = "region" in site_name.lower()
        color = "#888888"

    for c in df.columns:
        if c not in ["site", "site_name", "name"]:
            try:
                df[c] = pd.to_numeric(df[c], errors="ignore")
            except Exception:
                pass

    site_data[site_name] = df
    site_rows.append({
        "site": site_name,
        "lat": lat_val,
        "lon": lon_val,
        "central": central,
        "is_region": is_region,
        "color": color,
        "n_rows": len(df),
        "n_onset_raw": int(pd.Series(df.get(TARGET, pd.Series(dtype=int))).fillna(0).sum()) if TARGET in df.columns else np.nan,
        "has_ews": "ews_composite" in df.columns,
        "has_heat": "heat_lag4" in df.columns,
        "has_upwelling": "upwelling" in df.columns,
        "has_onset": TARGET in df.columns,
    })

site_meta_from_data = pd.DataFrame(site_rows).sort_values("lat", ascending=False).reset_index(drop=True)

print(f"Loaded {len(site_data)} site tables")
display_cols = ["site", "lat", "n_rows", "n_onset_raw", "has_ews", "has_heat", "has_upwelling", "has_onset"]
print(site_meta_from_data[display_cols].to_string(index=False))

all_col_sets = [set(df.columns) for df in site_data.values()]
common_cols = set.intersection(*all_col_sets) if all_col_sets else set()

BASELINE_FEATURES = [f for f in BASELINE_FEATURES_PREF if f in common_cols]
FULL_FEATURES = [f for f in FULL_FEATURES_PREF if f in common_cols]

if "heat_lag4" not in BASELINE_FEATURES:
    raise ValueError("heat_lag4 is required for the climate-only baseline.")
if "ews_composite" not in FULL_FEATURES:
    raise ValueError("ews_composite is required for the full model.")
if TARGET not in common_cols:
    raise ValueError(
        "onset is missing from at least one site table.\n"
        "Add an onset column, or include suppressed so this notebook can derive it."
    )

print("\nFeature sets actually used:")
print("  BASELINE_FEATURES =", BASELINE_FEATURES)
print("  FULL_FEATURES     =", FULL_FEATURES)
print(f"  Sites available   = {len(site_data)}")

In [ ]:
# ============================================================
# CELL 4 — HELPERS
# ============================================================
def block_bootstrap_auc(y_true, y_score, block_len=4, B=2000, seed=42):
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)

    if len(y_true) != len(y_score):
        raise ValueError("y_true and y_score must have equal length")
    if len(y_true) < block_len or np.unique(y_true).size < 2:
        return {"auc": np.nan, "ci_lo": np.nan, "ci_hi": np.nan, "n_boot": 0}

    n = len(y_true)
    starts = np.arange(0, n - block_len + 1)
    n_blocks = int(np.ceil(n / block_len))
    rng = np.random.default_rng(seed)

    aucs = []
    for _ in range(B):
        chosen = rng.choice(starts, size=n_blocks, replace=True)
        idx = np.concatenate([np.arange(s, s + block_len) for s in chosen])[:n]
        yb = y_true[idx]
        sb = y_score[idx]
        if np.unique(yb).size < 2:
            continue
        aucs.append(roc_auc_score(yb, sb))

    if len(aucs) == 0:
        return {"auc": np.nan, "ci_lo": np.nan, "ci_hi": np.nan, "n_boot": 0}

    aucs = np.asarray(aucs)
    return {
        "auc": float(np.mean(aucs)),
        "ci_lo": float(np.quantile(aucs, 0.025)),
        "ci_hi": float(np.quantile(aucs, 0.975)),
        "n_boot": int(len(aucs)),
    }

def paired_block_bootstrap_delta_auc(y_true, full_score, base_score, block_len=4, B=2000, seed=42):
    y_true = np.asarray(y_true, dtype=int)
    full_score = np.asarray(full_score, dtype=float)
    base_score = np.asarray(base_score, dtype=float)

    n = len(y_true)
    if n < block_len or np.unique(y_true).size < 2:
        return {"delta": np.nan, "ci_lo": np.nan, "ci_hi": np.nan, "n_boot": 0}

    starts = np.arange(0, n - block_len + 1)
    n_blocks = int(np.ceil(n / block_len))
    rng = np.random.default_rng(seed)

    deltas = []
    for _ in range(B):
        chosen = rng.choice(starts, size=n_blocks, replace=True)
        idx = np.concatenate([np.arange(s, s + block_len) for s in chosen])[:n]
        yb = y_true[idx]
        if np.unique(yb).size < 2:
            continue
        af = roc_auc_score(yb, full_score[idx])
        ab = roc_auc_score(yb, base_score[idx])
        deltas.append(af - ab)

    if len(deltas) == 0:
        return {"delta": np.nan, "ci_lo": np.nan, "ci_hi": np.nan, "n_boot": 0}

    deltas = np.asarray(deltas)
    return {
        "delta": float(np.mean(deltas)),
        "ci_lo": float(np.quantile(deltas, 0.025)),
        "ci_hi": float(np.quantile(deltas, 0.975)),
        "n_boot": int(len(deltas)),
    }

def next_quarter_warning_metrics(y_true, y_score, threshold=0.35):
    """
    Strict prospective evaluation:
    warning at quarter t counts only if onset occurs at quarter t+1
    """
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)

    if len(y_true) < 2:
        return {
            "tp": 0, "fa": 0, "fn": 0,
            "precision": np.nan, "recall": np.nan, "f1": np.nan,
            "eligible_onsets": 0, "warning_rate": np.nan,
        }

    warned_now = (y_score[:-1] >= threshold)
    onset_next = (y_true[1:] == 1)

    tp = int(np.sum(warned_now & onset_next))
    fa = int(np.sum(warned_now & ~onset_next))
    fn = int(np.sum(~warned_now & onset_next))

    precision = tp / (tp + fa) if (tp + fa) > 0 else np.nan
    recall = tp / np.sum(onset_next) if np.sum(onset_next) > 0 else np.nan
    f1 = (2 * precision * recall / (precision + recall)
          if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
          else np.nan)

    return {
        "tp": tp,
        "fa": fa,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "eligible_onsets": int(np.sum(onset_next)),
        "warning_rate": float(np.mean(warned_now)) if len(warned_now) else np.nan,
    }

def choose_train_sites(held_site, meta_df, site_data, mode="neighbor", neighbor_deg=3.0):
    others = [s for s in site_data.keys() if s != held_site]
    if mode == "loso":
        return others

    if held_site not in meta_df["site"].values:
        return others

    lat_h = float(meta_df.loc[meta_df["site"] == held_site, "lat"].iloc[0])
    if np.isnan(lat_h):
        return others

    neighbors = []
    for s in others:
        if s not in meta_df["site"].values:
            continue
        lat_s = float(meta_df.loc[meta_df["site"] == s, "lat"].iloc[0])
        if pd.notna(lat_s) and abs(lat_s - lat_h) <= neighbor_deg:
            neighbors.append(s)
    return neighbors

def one_sided_permutation_diff(x, g, B=10000, seed=42):
    """
    x = values, g = boolean group indicator
    tests H1: mean(x[g]) > mean(x[~g])
    """
    x = np.asarray(x, dtype=float)
    g = np.asarray(g, dtype=bool)

    valid = np.isfinite(x) & pd.notna(g)
    x = x[valid]
    g = g[valid]
    if g.sum() == 0 or (~g).sum() == 0:
        return np.nan, np.nan

    obs = float(x[g].mean() - x[~g].mean())
    rng = np.random.default_rng(seed)
    perm = []
    for _ in range(B):
        gs = rng.permutation(g)
        perm.append(float(x[gs].mean() - x[~gs].mean()))
    perm = np.asarray(perm)
    p = float(np.mean(perm >= obs))
    return obs, p

print("Helpers ready")

In [ ]:
# ============================================================
# CELL 5 — HELD-OUT SITE COMPARISON
# Climate-only vs Full model on the SAME held-out site
# ============================================================
results = []
oof_rows = []

meta_use = site_meta_from_data.copy()

print("=" * 88)
print("SITE-BY-SITE HELD-OUT COMPARISON")
print("=" * 88)
print(f"{'Site':<22} {'TrainSites':>10} {'TeOn':>4} {'BaseAUC':>8} {'FullAUC':>8} {'Delta':>8} {'Delta 95% CI':>24}  Note")
print("-" * 88)

for held_site, held_df_raw in site_data.items():
    held_df = held_df_raw.copy()

    needed_test = FULL_FEATURES + [TARGET]
    test_df = held_df[needed_test].dropna().copy()

    lat_val = meta_use.loc[meta_use["site"] == held_site, "lat"].iloc[0] if (meta_use["site"] == held_site).any() else np.nan
    lon_val = meta_use.loc[meta_use["site"] == held_site, "lon"].iloc[0] if (meta_use["site"] == held_site).any() else np.nan
    central = bool(meta_use.loc[meta_use["site"] == held_site, "central"].iloc[0]) if (meta_use["site"] == held_site).any() else False
    is_region = bool(meta_use.loc[meta_use["site"] == held_site, "is_region"].iloc[0]) if (meta_use["site"] == held_site).any() else False
    color = meta_use.loc[meta_use["site"] == held_site, "color"].iloc[0] if (meta_use["site"] == held_site).any() else "#888888"

    base_info = {
        "site": held_site,
        "lat": lat_val,
        "lon": lon_val,
        "central": central,
        "is_region": is_region,
        "color": color,
        "n_test_rows": len(test_df),
    }

    if len(test_df) < MIN_TEST_ROWS or test_df[TARGET].nunique() < 2:
        note = "skip: insufficient test variation"
        n_on = int(test_df[TARGET].sum()) if TARGET in test_df else 0
        print(f"{held_site:<22} {0:10d} {n_on:4d} {'—':>8} {'—':>8} {'—':>8} {'[skip]':>24}  {note}")
        results.append({
            **base_info,
            "train_sites": [],
            "n_train_sites": 0,
            "n_test_onset": n_on,
            "auc_base": np.nan,
            "auc_full": np.nan,
            "auc_base_ci_lo": np.nan,
            "auc_base_ci_hi": np.nan,
            "auc_full_ci_lo": np.nan,
            "auc_full_ci_hi": np.nan,
            "delta_auc": np.nan,
            "delta_ci_lo": np.nan,
            "delta_ci_hi": np.nan,
            "sig_delta": False,
            "ap_base": np.nan,
            "ap_full": np.nan,
            "brier_base": np.nan,
            "brier_full": np.nan,
            "tp_base": np.nan,
            "fa_base": np.nan,
            "fn_base": np.nan,
            "precision_base": np.nan,
            "recall_base": np.nan,
            "f1_base": np.nan,
            "tp_full": np.nan,
            "fa_full": np.nan,
            "fn_full": np.nan,
            "precision_full": np.nan,
            "recall_full": np.nan,
            "f1_full": np.nan,
            "skip_reason": note,
        })
        continue

    train_sites = choose_train_sites(held_site, meta_use, site_data, mode=CV_MODE, neighbor_deg=NEIGHBOR_DEG)

    train_frames = []
    for s in train_sites:
        sdf = site_data[s].copy()
        tmp = sdf[FULL_FEATURES + [TARGET]].dropna().copy()
        if len(tmp):
            train_frames.append(tmp)

    if len(train_frames) == 0:
        note = "skip: no eligible training data"
        print(f"{held_site:<22} {len(train_sites):10d} {int(test_df[TARGET].sum()):4d} {'—':>8} {'—':>8} {'—':>8} {'[skip]':>24}  {note}")
        results.append({
            **base_info,
            "train_sites": train_sites,
            "n_train_sites": len(train_sites),
            "n_test_onset": int(test_df[TARGET].sum()),
            "skip_reason": note,
        })
        continue

    train_df = pd.concat(train_frames, axis=0)

    if train_df[TARGET].sum() < MIN_TRAIN_ONSETS or train_df[TARGET].nunique() < 2:
        note = "skip: too few train onsets"
        print(f"{held_site:<22} {len(train_sites):10d} {int(test_df[TARGET].sum()):4d} {'—':>8} {'—':>8} {'—':>8} {'[skip]':>24}  {note}")
        results.append({
            **base_info,
            "train_sites": train_sites,
            "n_train_sites": len(train_sites),
            "n_test_onset": int(test_df[TARGET].sum()),
            "skip_reason": note,
        })
        continue

    y_tr = train_df[TARGET].astype(int).values
    y_te = test_df[TARGET].astype(int).values

    sc_b = StandardScaler()
    Xb_tr = sc_b.fit_transform(train_df[BASELINE_FEATURES].values)
    Xb_te = sc_b.transform(test_df[BASELINE_FEATURES].values)
    lr_b = LogisticRegression(C=0.5, class_weight="balanced", max_iter=1000, random_state=42)
    lr_b.fit(Xb_tr, y_tr)
    prob_b = lr_b.predict_proba(Xb_te)[:, 1]

    sc_f = StandardScaler()
    Xf_tr = sc_f.fit_transform(train_df[FULL_FEATURES].values)
    Xf_te = sc_f.transform(test_df[FULL_FEATURES].values)
    lr_f = LogisticRegression(C=0.5, class_weight="balanced", max_iter=1000, random_state=42)
    lr_f.fit(Xf_tr, y_tr)
    prob_f = lr_f.predict_proba(Xf_te)[:, 1]

    auc_base = roc_auc_score(y_te, prob_b)
    auc_full = roc_auc_score(y_te, prob_f)

    bb_delta = paired_block_bootstrap_delta_auc(y_te, prob_f, prob_b, block_len=BLOCK_LEN, B=B_BOOT)

    ap_base = average_precision_score(y_te, prob_b)
    ap_full = average_precision_score(y_te, prob_f)
    brier_base = brier_score_loss(y_te, prob_b)
    brier_full = brier_score_loss(y_te, prob_f)

    warn_b = next_quarter_warning_metrics(y_te, prob_b, threshold=THRESHOLD)
    warn_f = next_quarter_warning_metrics(y_te, prob_f, threshold=THRESHOLD)

    sig_delta = pd.notna(bb_delta["ci_lo"]) and (bb_delta["ci_lo"] > 0)
    ci_str = f"[{bb_delta['ci_lo']:.3f}, {bb_delta['ci_hi']:.3f}]" if pd.notna(bb_delta["ci_lo"]) else "[N/A]"
    note = "★ EWS adds value" if sig_delta else ""

    print(
        f"{held_site:<22} {len(train_sites):10d} {int(y_te.sum()):4d} "
        f"{auc_base:8.3f} {auc_full:8.3f} {auc_full - auc_base:8.3f} "
        f"{ci_str:>24}  {note}"
    )

    results.append({
        **base_info,
        "train_sites": train_sites,
        "n_train_sites": len(train_sites),
        "n_train_onset": int(y_tr.sum()),
        "n_test_onset": int(y_te.sum()),
        "auc_base": auc_base,
        "auc_full": auc_full,
        "auc_base_ci_lo": np.nan,
        "auc_base_ci_hi": np.nan,
        "auc_full_ci_lo": np.nan,
        "auc_full_ci_hi": np.nan,
        "delta_auc": auc_full - auc_base,
        "delta_ci_lo": bb_delta["ci_lo"],
        "delta_ci_hi": bb_delta["ci_hi"],
        "sig_delta": sig_delta,
        "ap_base": ap_base,
        "ap_full": ap_full,
        "brier_base": brier_base,
        "brier_full": brier_full,
        "tp_base": warn_b["tp"],
        "fa_base": warn_b["fa"],
        "fn_base": warn_b["fn"],
        "precision_base": warn_b["precision"],
        "recall_base": warn_b["recall"],
        "f1_base": warn_b["f1"],
        "tp_full": warn_f["tp"],
        "fa_full": warn_f["fa"],
        "fn_full": warn_f["fn"],
        "precision_full": warn_f["precision"],
        "recall_full": warn_f["recall"],
        "f1_full": warn_f["f1"],
        "warning_rate_base": warn_b["warning_rate"],
        "warning_rate_full": warn_f["warning_rate"],
        "skip_reason": "",
    })

    pred_df = pd.DataFrame({
        "site": held_site,
        "time": test_df.index,
        "y": y_te,
        "prob_base": prob_b,
        "prob_full": prob_f,
        "lat": lat_val,
        "lon": lon_val,
        "central": central,
        "is_region": is_region,
    })
    oof_rows.append(pred_df)

res_df = pd.DataFrame(results).sort_values("lat", ascending=False).reset_index(drop=True)
oof_df = pd.concat(oof_rows, axis=0, ignore_index=True) if len(oof_rows) else pd.DataFrame()

res_path = OUTPUT_DIR / "site_value_add_results.csv"
res_df.to_csv(res_path, index=False)
print("\nSaved site results →", res_path)

if len(oof_df):
    oof_path = OUTPUT_DIR / "oof_predictions.csv"
    oof_df.to_csv(oof_path, index=False)
    print("Saved held-out predictions →", oof_path)

display_cols = [
    "site", "lat", "n_test_onset", "auc_base", "auc_full",
    "delta_auc", "delta_ci_lo", "delta_ci_hi", "sig_delta"
]
print("\nResult preview:")
print(res_df[display_cols].to_string(index=False))

In [ ]:
# ============================================================
# CELL 6 — SUMMARY STATS + GEOGRAPHIC TESTS
# ============================================================
valid = res_df.dropna(subset=["delta_auc"]).copy()

if valid.empty:
    raise RuntimeError("No valid held-out folds were produced. Check SITE_DATA_PATH and feature columns.")

central_vals = valid.loc[valid["central"], "delta_auc"].values
noncentral_vals = valid.loc[~valid["central"], "delta_auc"].values

mwu_p = np.nan
if len(central_vals) >= 2 and len(noncentral_vals) >= 2:
    _, mwu_p = mannwhitneyu(central_vals, noncentral_vals, alternative="greater")

obs_diff, perm_p = one_sided_permutation_diff(valid["delta_auc"].values, valid["central"].values, B=5000)

site_summary = {
    "n_valid_sites": len(valid),
    "n_sig_delta": int(valid["sig_delta"].sum()),
    "mean_auc_base": float(valid["auc_base"].mean()),
    "mean_auc_full": float(valid["auc_full"].mean()),
    "mean_delta_auc": float(valid["delta_auc"].mean()),
    "median_delta_auc": float(valid["delta_auc"].median()),
    "central_mean_delta": float(np.nanmean(central_vals)) if len(central_vals) else np.nan,
    "noncentral_mean_delta": float(np.nanmean(noncentral_vals)) if len(noncentral_vals) else np.nan,
    "central_minus_noncentral": obs_diff,
    "mwu_p": mwu_p,
    "perm_p": perm_p,
}

print("=" * 72)
print("HEADLINE SUMMARY")
print("=" * 72)
print(f"Valid held-out sites:            {site_summary['n_valid_sites']}")
print(f"Sites with positive sig ΔAUC:    {site_summary['n_sig_delta']}")
print(f"Mean climate-only AUC:           {site_summary['mean_auc_base']:.3f}")
print(f"Mean full-model AUC:             {site_summary['mean_auc_full']:.3f}")
print(f"Mean ΔAUC (full - climate):      {site_summary['mean_delta_auc']:.3f}")
print(f"Median ΔAUC:                     {site_summary['median_delta_auc']:.3f}")
print()

if len(central_vals) and len(noncentral_vals):
    print("GEOGRAPHIC ENRICHMENT TEST")
    print(f"Central CA mean ΔAUC:            {site_summary['central_mean_delta']:.3f}  (n={len(central_vals)})")
    print(f"Non-central mean ΔAUC:           {site_summary['noncentral_mean_delta']:.3f}  (n={len(noncentral_vals)})")
    print(f"Difference:                      {site_summary['central_minus_noncentral']:.3f}")
    if pd.notna(mwu_p):
        print(f"Mann–Whitney p (one-sided):      {mwu_p:.4f}")
    print(f"Permutation p (5,000 shuffles): {perm_p:.4f}")
    print()

if len(oof_df):
    y_pool = oof_df["y"].astype(int).values
    pb_pool = oof_df["prob_base"].values
    pf_pool = oof_df["prob_full"].values

    pooled = {
        "auc_base": roc_auc_score(y_pool, pb_pool) if np.unique(y_pool).size > 1 else np.nan,
        "auc_full": roc_auc_score(y_pool, pf_pool) if np.unique(y_pool).size > 1 else np.nan,
        "ap_base": average_precision_score(y_pool, pb_pool),
        "ap_full": average_precision_score(y_pool, pf_pool),
        "brier_base": brier_score_loss(y_pool, pb_pool),
        "brier_full": brier_score_loss(y_pool, pf_pool),
    }

    pooled_warn_base = next_quarter_warning_metrics(y_pool, pb_pool, threshold=THRESHOLD)
    pooled_warn_full = next_quarter_warning_metrics(y_pool, pf_pool, threshold=THRESHOLD)

    print("=" * 72)
    print("POOLED HELD-OUT FORECAST UTILITY")
    print("=" * 72)
    print(f"Pooled AUC     | base={pooled['auc_base']:.3f}  full={pooled['auc_full']:.3f}  Δ={pooled['auc_full']-pooled['auc_base']:.3f}")
    print(f"Pooled AP      | base={pooled['ap_base']:.3f}  full={pooled['ap_full']:.3f}")
    print(f"Pooled Brier   | base={pooled['brier_base']:.3f}  full={pooled['brier_full']:.3f}  (lower is better)")
    print()
    print("Strict next-quarter warning metrics (threshold = {:.2f})".format(THRESHOLD))
    print(f"Precision      | base={pooled_warn_base['precision']:.3f}  full={pooled_warn_full['precision']:.3f}")
    print(f"Recall         | base={pooled_warn_base['recall']:.3f}  full={pooled_warn_full['recall']:.3f}")
    print(f"F1             | base={pooled_warn_base['f1']:.3f}  full={pooled_warn_full['f1']:.3f}")
    print(f"False alarms   | base={pooled_warn_base['fa']}  full={pooled_warn_full['fa']}")
    print(f"Warning rate   | base={pooled_warn_base['warning_rate']:.3f}  full={pooled_warn_full['warning_rate']:.3f}")
else:
    pooled = {}
    pooled_warn_base = {}
    pooled_warn_full = {}

if RUN_LOW_N_ROBUSTNESS:
    high_n = valid[valid["n_test_onset"] >= LOW_N_MIN_EVENTS].copy()
    if len(high_n) >= 5:
        c_hi = high_n.loc[high_n["central"], "delta_auc"].values
        nc_hi = high_n.loc[~high_n["central"], "delta_auc"].values
        hi_diff, hi_perm_p = one_sided_permutation_diff(high_n["delta_auc"].values, high_n["central"].values, B=3000)

        print()
        print("=" * 72)
        print(f"LOW-N ROBUSTNESS (restricted to sites with n_onset ≥ {LOW_N_MIN_EVENTS})")
        print("=" * 72)
        print(f"Eligible sites:                 {len(high_n)}")
        print(f"Mean ΔAUC (restricted):         {high_n['delta_auc'].mean():.3f}")
        if len(c_hi) and len(nc_hi):
            print(f"Central mean ΔAUC:              {np.mean(c_hi):.3f}")
            print(f"Non-central mean ΔAUC:          {np.mean(nc_hi):.3f}")
            print(f"Central - noncentral:           {hi_diff:.3f}")
            print(f"Permutation p:                  {hi_perm_p:.4f}")
    else:
        print(f"\nLow-n robustness skipped: fewer than 5 sites with n_onset ≥ {LOW_N_MIN_EVENTS}")

summary_json = OUTPUT_DIR / "headline_summary.json"
with open(summary_json, "w", encoding="utf-8") as f:
    json.dump(site_summary, f, indent=2)
print("\nSaved headline summary →", summary_json)

In [ ]:
# ============================================================
# CELL 7 — FIGURE 1
# Panel A: California map of ΔAUC
# Panel B: Paired site-by-site baseline vs full AUC
# ============================================================
plot_df = valid.sort_values("lat", ascending=False).reset_index(drop=True)

CA_COAST_LON = [
    -124.21, -124.35, -124.56, -124.61, -124.51, -124.41, -124.21,
    -124.07, -123.82, -123.69, -123.44, -123.11, -122.88, -122.61,
    -122.47, -122.39, -122.17, -121.90, -121.55, -121.29, -120.89,
    -120.67, -120.47, -120.17, -119.87, -119.53, -119.23, -118.98,
    -118.56, -118.29, -118.00, -117.67, -117.42, -117.25, -117.12
]
CA_COAST_LAT = [
    41.98, 41.72, 41.31, 40.83, 40.44, 40.04, 39.73,
    39.36, 38.97, 38.54, 38.24, 37.95, 37.75, 37.52,
    37.21, 36.97, 36.60, 36.27, 35.82, 35.42, 35.15,
    34.90, 34.52, 34.26, 34.05, 33.89, 33.72, 33.48,
    33.22, 32.98, 32.76, 32.63, 32.54, 32.53, 32.54
]

fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor("#f7f9fc")
gs = fig.add_gridspec(1, 2, width_ratios=[1.15, 1.0], wspace=0.18)

ax_map = fig.add_subplot(gs[0, 0])
ax_map.set_facecolor("#d0e8f5")

land_lon = CA_COAST_LON + [-128, -128, CA_COAST_LON[0]]
land_lat = CA_COAST_LAT + [CA_COAST_LAT[0], CA_COAST_LAT[-1], CA_COAST_LAT[0]]
ax_map.fill(land_lon, land_lat, color="#e8e0d0", zorder=1, alpha=0.95)
ax_map.plot(CA_COAST_LON, CA_COAST_LAT, color="#8b7355", lw=1.6, zorder=2)

ax_map.axhspan(36.5, 38.5, alpha=0.16, color="#e74c3c", zorder=0)
ax_map.axhline(36.5, color="#e74c3c", lw=0.8, ls="--", alpha=0.5, zorder=2)
ax_map.axhline(38.5, color="#e74c3c", lw=0.8, ls="--", alpha=0.5, zorder=2)

dmin = float(plot_df["delta_auc"].min())
dmax = float(plot_df["delta_auc"].max())
if dmin < 0 < dmax:
    norm = TwoSlopeNorm(vmin=dmin, vcenter=0.0, vmax=dmax)
else:
    norm = Normalize(vmin=dmin, vmax=dmax)
cmap = plt.cm.coolwarm

for _, row in plot_df.iterrows():
    if pd.isna(row["lon"]) or pd.isna(row["lat"]):
        continue

    size = np.clip(90 + 260 * max(row["auc_full"] - 0.45, 0), 70, 320)
    face = cmap(norm(row["delta_auc"]))
    marker = "*" if row["is_region"] else "o"

    if row["sig_delta"]:
        ax_map.scatter(
            row["lon"], row["lat"], s=size * 2.0,
            facecolor="none", edgecolor="black", lw=1.8, zorder=4, marker=marker
        )

    ax_map.scatter(
        row["lon"], row["lat"], s=size,
        color=face, edgecolor="white", lw=0.8, alpha=0.95, zorder=5, marker=marker
    )

    x_off = 0.35 if row["lon"] > -120.5 else -0.35
    ha = "left" if x_off > 0 else "right"
    label = row["site"].replace(" Region", " (R)")
    ax_map.text(
        row["lon"] + x_off, row["lat"], label,
        fontsize=6.8, ha=ha, va="center", zorder=6, color="#1a1a1a",
        path_effects=[pe.withStroke(linewidth=1.5, foreground="white")]
    )

for regime_name, lat_lo, lat_hi, col in [
    ("NorCal", 40.0, 43.0, "#1f77b4"),
    ("Central CA", 36.5, 40.0, "#d6604d"),
    ("Big Sur", 34.8, 36.5, "#9467bd"),
    ("SoCal", 32.0, 34.8, "#2ca02c"),
]:
    mask = (plot_df["lat"] >= lat_lo) & (plot_df["lat"] < lat_hi)
    n_sig = int(plot_df.loc[mask, "sig_delta"].sum())
    n_tot = int(mask.sum())
    ax_map.text(
        -127.6, (lat_lo + lat_hi) / 2.0, f"{n_sig}/{n_tot}\nΔAUC sig",
        fontsize=7, ha="center", va="center", color=col, fontweight="bold",
        path_effects=[pe.withStroke(linewidth=2, foreground="white")]
    )

ax_map.set_xlim(-128.5, -116.5)
ax_map.set_ylim(32.0, 42.5)
ax_map.set_xlabel("Longitude (°W)")
ax_map.set_ylabel("Latitude (°N)")
ax_map.set_title(
    "A. California Map — Where Does EWS Add Value?\n"
    "Marker color = ΔAUC (Full − Climate-only) | Black ring = ΔAUC CI > 0",
    fontweight="bold"
)
ax_map.grid(True, alpha=0.18, color="white", zorder=0)

sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cb = plt.colorbar(sm, ax=ax_map, fraction=0.035, pad=0.02)
cb.set_label("ΔAUC (Full − Climate-only)")

legend_elems = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=cmap(norm(max(dmax, 0))),
           markersize=8, label="Positive ΔAUC"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=cmap(norm(min(dmin, 0))),
           markersize=8, label="Negative ΔAUC"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="gray",
           markeredgecolor="black", markeredgewidth=1.5, markersize=8,
           label="ΔAUC significant"),
    Line2D([0], [0], marker="*", color="w", markerfacecolor="gray",
           markersize=10, label="Region-level site"),
]
ax_map.legend(handles=legend_elems, fontsize=8, loc="lower left", framealpha=0.9)

ax_pair = fig.add_subplot(gs[0, 1])
ax_pair.set_facecolor("white")

for i, row in plot_df.iterrows():
    y = i
    line_col = "#2c3e50" if row["delta_auc"] >= 0 else "#95a5a6"
    ax_pair.plot(
        [row["auc_base"], row["auc_full"]], [y, y],
        color=line_col, lw=2.0, alpha=0.75, zorder=1
    )

    ax_pair.scatter(
        row["auc_base"], y, s=55, color="#95a5a6", edgecolor="white",
        lw=0.8, zorder=3, label="Climate-only" if i == 0 else None
    )

    face = "#27ae60" if row["sig_delta"] else row["color"]
    ax_pair.scatter(
        row["auc_full"], y,
        s=85 if row["sig_delta"] else 65,
        color=face, edgecolor="black" if row["sig_delta"] else "white",
        lw=1.0, zorder=4, label="Full model" if i == 0 else None
    )

    if row["sig_delta"]:
        ax_pair.text(
            max(row["auc_base"], row["auc_full"]) + 0.02, y, "★",
            va="center", fontsize=12, color="#e67e22"
        )

ax_pair.axvline(0.5, ls="--", color="#aaaaaa", lw=1.2)
ax_pair.set_yticks(range(len(plot_df)))
ax_pair.set_yticklabels([f"{r.site} ({r.lat:.1f}°N)" for _, r in plot_df.iterrows()], fontsize=8.5)
ax_pair.set_xlabel("Held-out AUC")
ax_pair.set_title(
    "B. Same Site, Same Quarters: Climate-only vs Full Model\n"
    "Green/outlined full-model points = significant positive ΔAUC",
    fontweight="bold"
)
ax_pair.legend(fontsize=8, loc="lower right")
ax_pair.set_xlim(
    max(0.0, float(plot_df[["auc_base", "auc_full"]].min().min()) - 0.08),
    min(1.05, float(plot_df[["auc_base", "auc_full"]].max().max()) + 0.15)
)

fig.suptitle(
    f"Spatial Value-Add Atlas for Kelp EWS  |  Held-out protocol = {CV_MODE.upper()}",
    fontsize=13, fontweight="bold", y=0.98
)

fig.tight_layout()
fig1_path = OUTPUT_DIR / "figure1_spatial_value_add_atlas.png"
fig.savefig(fig1_path, dpi=220, bbox_inches="tight", facecolor="#f7f9fc")
plt.show()
print("Saved →", fig1_path)

In [ ]:
# ============================================================
# CELL 8 — FIGURE 2
# Panel A: Central vs non-central ΔAUC
# Panel B: Calibration curve (pooled held-out predictions)
# ============================================================
fig = plt.figure(figsize=(14, 6))
fig.patch.set_facecolor("#f7f9fc")
gs = fig.add_gridspec(1, 2, width_ratios=[0.95, 1.05], wspace=0.28)

ax_a = fig.add_subplot(gs[0, 0])
ax_a.set_facecolor("white")

groups = [
    valid.loc[valid["central"], "delta_auc"].dropna().values,
    valid.loc[~valid["central"], "delta_auc"].dropna().values,
]
labels = [
    f"Central CA\n(36.5–38.5°N)\nn={len(groups[0])}",
    f"Non-central\nn={len(groups[1])}",
]

bp = ax_a.boxplot(
    groups, labels=labels, patch_artist=True,
    boxprops=dict(alpha=0.75),
    medianprops=dict(color="black", linewidth=2.2),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2),
)
bp["boxes"][0].set_facecolor("#d6604d")
bp["boxes"][1].set_facecolor("#95a5a6")

rng = np.random.default_rng(42)
for i, vals in enumerate(groups, start=1):
    jitter = rng.uniform(-0.08, 0.08, len(vals))
    ax_a.scatter(np.full(len(vals), i) + jitter, vals, s=35, color="#2c3e50", alpha=0.75, zorder=4)

ax_a.axhline(0.0, ls="--", color="#aaaaaa", lw=1.2)
ax_a.set_ylabel("ΔAUC (Full − Climate-only)")
title_line = f"Mann–Whitney p={mwu_p:.4f}" if pd.notna(mwu_p) else "Mann–Whitney p=N/A"
ax_a.set_title(
    "A. Is EWS Value-Add Concentrated in Central California?\n"
    f"{title_line}  |  Permutation p={perm_p:.4f}",
    fontweight="bold"
)

ax_b = fig.add_subplot(gs[0, 1])
ax_b.set_facecolor("white")

if len(oof_df):
    y_pool = oof_df["y"].astype(int).values

    for label, col_name, color in [
        ("Climate-only", "prob_base", "#7f8c8d"),
        ("Full model", "prob_full", "#2980b9"),
    ]:
        p_arr = oof_df[col_name].values
        try:
            frac_pos, mean_pred = calibration_curve(y_pool, p_arr, n_bins=6, strategy="quantile")
            brier = brier_score_loss(y_pool, p_arr)
            ax_b.plot(mean_pred, frac_pos, "o-", color=color, lw=2, ms=7, label=f"{label} (Brier={brier:.3f})")
        except Exception as e:
            ax_b.text(
                0.5, 0.5, f"Calibration failed for {label}\n{e}",
                ha="center", va="center", transform=ax_b.transAxes
            )

    ax_b.plot([0, 1], [0, 1], "--", color="#555555", alpha=0.7, label="Perfect calibration")
    ax_b.set_xlim(0, 1)
    ax_b.set_ylim(0, 1)
    ax_b.set_xlabel("Mean predicted probability")
    ax_b.set_ylabel("Observed onset frequency")
    ax_b.set_title(
        "B. Pooled Held-Out Calibration\n"
        "Are the probabilities themselves meaningful?",
        fontweight="bold"
    )
    ax_b.legend(fontsize=8, loc="upper left")
else:
    ax_b.text(0.5, 0.5, "No pooled predictions available", ha="center", va="center", transform=ax_b.transAxes)

fig.suptitle("Secondary Rigor Checks", fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
fig2_path = OUTPUT_DIR / "figure2_geography_and_calibration.png"
fig.savefig(fig2_path, dpi=220, bbox_inches="tight", facecolor="#f7f9fc")
plt.show()
print("Saved →", fig2_path)

In [ ]:
# ============================================================
# CELL 9 — JUDGE-READY TAKEAWAYS
# ============================================================
valid_sorted = valid.sort_values("delta_auc", ascending=False).copy()

print("=" * 78)
print("JUDGE-READY INTERPRETATION")
print("=" * 78)

print("1) DOES EWS ADD VALUE BEYOND CLIMATE-ONLY DRIVERS?")
print(
    f"   Mean site AUC improved from {site_summary['mean_auc_base']:.3f} "
    f"to {site_summary['mean_auc_full']:.3f} "
    f"(Δ={site_summary['mean_delta_auc']:.3f})."
)
print(
    f"   Positive significant ΔAUC occurred at "
    f"{int(valid['sig_delta'].sum())} of {len(valid)} valid held-out sites."
)

if len(valid_sorted):
    top = valid_sorted.iloc[0]
    print(
        f"   Strongest site-level gain: {top['site']} "
        f"(ΔAUC={top['delta_auc']:.3f}, CI=[{top['delta_ci_lo']:.3f}, {top['delta_ci_hi']:.3f}])."
    )

print()
print("2) IS THE VALUE-ADD GEOGRAPHICALLY STRUCTURED?")
if len(central_vals) and len(noncentral_vals):
    print(
        f"   Central CA mean ΔAUC = {site_summary['central_mean_delta']:.3f}; "
        f"non-central mean ΔAUC = {site_summary['noncentral_mean_delta']:.3f}."
    )
    print(
        f"   Central − noncentral difference = {site_summary['central_minus_noncentral']:.3f} "
        f"(permutation p={site_summary['perm_p']:.4f})."
    )
    if pd.notna(site_summary["mwu_p"]):
        print(f"   Mann–Whitney one-sided p = {site_summary['mwu_p']:.4f}.")
else:
    print("   Not enough sites in one of the groups for a stable geographic comparison.")

print()
print("3) ARE THE FORECASTS USEFUL, NOT JUST RANKED WELL?")
if len(oof_df):
    print(
        f"   Pooled AP: base={pooled['ap_base']:.3f}, full={pooled['ap_full']:.3f}. "
        f"Pooled Brier: base={pooled['brier_base']:.3f}, full={pooled['brier_full']:.3f}."
    )
    print(
        f"   Strict next-quarter recall: base={pooled_warn_base['recall']:.3f}, "
        f"full={pooled_warn_full['recall']:.3f}; "
        f"precision: base={pooled_warn_base['precision']:.3f}, "
        f"full={pooled_warn_full['precision']:.3f}."
    )

print()
print("4) WHAT YOU CAN SAY TO JUDGES")
print("   “I did not just show that my full model has decent AUC.")
print("    I tested whether EWS adds value beyond climate-only drivers on the same held-out site,")
print("    using paired block-bootstrap confidence intervals. Then I mapped where that added value appears.")
print("    If the strongest gains cluster in central California, that supports a mechanistic story")
print("    that EWS is most informative in the upwelling-dominated regime.”")

if RUN_LOW_N_ROBUSTNESS:
    print()
    print("5) LOW-N ROBUSTNESS")
    print(
        f"   I also checked the pattern after restricting to sites with at least "
        f"{LOW_N_MIN_EVENTS} onset events."
    )
    print("   That helps defend against the critique that one-event sites are driving the pattern.")

print("=" * 78)